# 18. Agents SDK 대조

직접 만든 하네스를 OpenAI Agents SDK로 옮겨 프레임워크가 대신하거나 감추는 부분을 비교한다.

### 공통 설정

Part 1 `1`과 같은 블록이다. 노트북끼리 코드를 공유하지 않는다.

In [1]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]

client = OpenAI()

PRICES = {MODEL: (2.50, 15.00)}  # 1M 토큰당 (입력, 출력) USD
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

코퍼스와 도구 함수, 이 노트북의 과제다. 도구는 `grep_docs` 하나만 쓴다.

In [2]:
CORPUS = Path("../../../../../data/corpus/graph")
TASK = "'한국독립당' 이 나오는 문서가 몇 편인지 세고 숫자로 답한다"


def grep_docs(pattern):
    return [{"title": f.stem, "count": f.read_text(encoding="utf-8").count(pattern)}
            for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]


print(len(grep_docs("한국독립당")), "편")

16 편


### 설치 주의

`openai-agents`는 `openai` 3.x를 요구하지만 이 과정은 `openai==2.52.0`으로 고정되어 있어 `pip`이 충돌을 경고한다. 이 회차 전용 가상환경에서 사용한다.

### 자작 하네스

도구 정의와 설정을 먼저 둔다. 도구 오류는 예외 대신 문자열로 되돌린다.

In [3]:
TOOLS = [{"type": "function", "name": "grep_docs", "strict": True,
          "description": "낱말이 든 문서의 제목과 등장 횟수를 돌려준다.",
          "parameters": {"type": "object",
                         "properties": {"pattern": {"type": "string", "description": "찾을 낱말"}},
                         "required": ["pattern"], "additionalProperties": False}}]

CONFIG = {"instructions": "도구로 확인한 것만 답한다.", "max_turns": 6,
          "max_tokens": 60000, "tool_error": "error: {e}"}


def call_tool(name, args):
    try:
        return json.dumps(grep_docs(**args), ensure_ascii=False)
    except Exception as e:
        return CONFIG["tool_error"].format(e=e)

루프는 Part 1 `1`과 같다. 입력 배열을 직접 조립하고 턴마다 부른 도구와 누적 토큰을 트레이스에 남긴다.

In [4]:
def run(task):
    messages = [{"role": "user", "content": task}]
    trace, used = [], 0
    for turn in range(1, CONFIG["max_turns"] + 1):
        r = keep(client.responses.create(model=MODEL, instructions=CONFIG["instructions"],
                                         input=messages, tools=TOOLS, store=False))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls], "토큰": used})
        if not calls:
            return r.output_text, trace
        if used > CONFIG["max_tokens"]:
            return None, trace
        messages += [i.model_dump() for i in r.output]
        for c in calls:
            messages.append({"type": "function_call_output", "call_id": c.call_id,
                             "output": call_tool(c.name, json.loads(c.arguments))})
    return None, trace

과제를 돌린다. 답과 트레이스가 따로 돌아온다.

In [5]:
answer, trace = run(TASK)
print(answer)

16


트레이스에는 턴마다 부른 도구와 누적 토큰이 남는다.

In [6]:
for row in trace:
    print(f"  턴 {row['턴']}  {row['토큰']:6,d} 토큰  {row['도구']}")

  턴 1     119 토큰  ['grep_docs']
  턴 2     503 토큰  []


### 같은 것을 SDK로

`@function_tool`은 타입 힌트와 독스트링으로 스키마를 만든다. 에이전트는 지시문과 도구 목록을 갖는다.
</br>어떤 종류의 function이든 상관없이 agent의 툴로 사용 가능

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool
def grep_docs_tool(pattern: str) -> list[dict]:
    """낱말이 든 문서의 제목과 등장 횟수를 돌려준다.""" # llm에게 주는 description
    return grep_docs(pattern)


counter = Agent(name="counter", model=MODEL,
                instructions="도구로 확인한 것만 답한다.", tools=[grep_docs_tool])

`Runner.run`이 실행 루프와 도구 결과 전달을 대신한다.
</br>Agent를 밖에서 돌려주는 Runner가 있음

In [ ]:
result = await Runner.run(counter, TASK)
print(result.final_output)

16


실행 중 생긴 아이템을 읽는다. 도구 호출과 결과가 아이템으로 남는다. (tool call, 결과, 메시지)

In [9]:
print([type(i).__name__ for i in result.new_items])

['ToolCallItem', 'ToolCallOutputItem', 'MessageOutputItem']


### 하네스 구성 요소의 대응

| 자작 하네스 | Agents SDK | 차이 |
|---|---|---|
| Runtime (`run`의 for 문) | `Runner.run` | 직접 만든 루프는 정지 조건이 셋이고 SDK는 `max_turns` 하나다 |
| ToolRegistry (`TOOLS`, `IMPL`) | `@function_tool` | 타입 힌트와 독스트링으로 스키마를 만든다 |
| Session (`messages` 조립) | `Session`, `result.to_input_list()` | 입력 내용을 명시적으로 꺼내야 확인할 수 있다 |
| Trace (`trace` 리스트) | `result.new_items`, 대시보드 | 스팬을 직접 정의할 수 없다 |
| Policy (`CONFIG["gated"]`) | `input_guardrail`, `output_guardrail` | 도구 단위가 아니라 실행 전후에 적용된다 |

도구 단위 승인이 필요하면 도구 구현에서 별도로 처리해야 한다.

### 컨텍스트 확인

`to_input_list`로 컨텍스트를 확인할 수 있지만 직접 변경할 수는 없다. 압축이나 오프로딩이 필요하면 `Session`을 구현해야 한다.

In [11]:
items = result.to_input_list()
print(f"입력 배열 {len(items)}개")
for it in items:
    kind = it.get("type") or it.get("role")
    body = json.dumps(it, ensure_ascii=False)
    print(f"  {str(kind):22} {len(body):5,d}자  {body}")

입력 배열 4개
  user                      65자  {"content": "'한국독립당' 이 나오는 문서가 몇 편인지 세고 숫자로 답한다", "role": "user"}
  function_call            221자  {"arguments": "{\"pattern\":\"한국독립당\"}", "call_id": "call_Dsnb3CX2gUMCZVQ9FyR6JqOp", "name": "grep_docs_tool", "type": "function_call", "id": "fc_056be3a48300512b006ad1c5f6238c87d0a9b664d6f4e466e2", "status": "completed"}
  function_call_output     611자  {"call_id": "call_Dsnb3CX2gUMCZVQ9FyR6JqOp", "output": "[{'title': '김구', 'count': 29}, {'title': '김규식', 'count': 15}, {'title': '김학규_(1900년)', 'count': 4}, {'title': '대한민국_임시정부', 'count': 36}, {'title': '박열', 'count': 1}, {'title': '안공근', 'count': 4}, {'title': '안공근_암살_사건', 'count': 3}, {'title': '안재홍', 'count': 4}, {'title': '안중근', 'count': 1}, {'title': '안창호', 'count': 10}, {'title': '오광선', 'count': 3}, {'title': '이봉창_의거', 'count': 2}, {'title': '이승만', 'count': 8}, {'title': '조선민족혁명당', 'count': 4}, {'title': '조영원_(1891년)', 'count': 13}, {'title': '한국독립당', 'count': 33}]", "type": "function_call_o

### 정지 조건 비교

SDK는 `max_turns`만 제공한다. 토큰이나 시간 제한은 실행 바깥에서 처리해야 한다.

In [12]:
import asyncio

from agents import MaxTurnsExceeded


async def run_sdk_budgeted(agent, task, max_turns=6, max_seconds=30):
    t0 = time.perf_counter()
    try:
        out = await asyncio.wait_for(Runner.run(agent, task, max_turns=max_turns),
                                     timeout=max_seconds)   # 비동기(async)지만 await으로 기다림
        return out.final_output, "완료", round(time.perf_counter() - t0, 1)
    except MaxTurnsExceeded:
        return None, "반복 예산", round(time.perf_counter() - t0, 1)
    except asyncio.TimeoutError:
        return None, "시간 예산", round(time.perf_counter() - t0, 1)

기본 예산으로 돌린다.

In [13]:
print(await run_sdk_budgeted(counter, TASK))

('16', '완료', 2.7)


반복 예산을 1로 좁힌다. 예외가 정지 사유로 바뀌어 돌아온다.

In [14]:
print(await run_sdk_budgeted(counter, TASK, max_turns=1))

(None, '반복 예산', 1.4)


SDK의 반복 한도 초과는 예외이므로 태스크셋 전체를 중단할 수 있다. 평가를 계속하려면 예외를 행 단위 결과로 변환해야 한다.

### 핸드오프와 가드레일

가드레일은 도구 단위가 아니라 실행 전후에 걸린다. 범위 밖 낱말이 들어오면 트립와이어를 올린다.

In [ ]:
from agents import GuardrailFunctionOutput, InputGuardrailTripwireTriggered, input_guardrail


@input_guardrail
async def scope_guard(ctx, agent, user_input):
    bad = any(w in str(user_input) for w in ("api key", "비밀번호", "날씨"))
    return GuardrailFunctionOutput(output_info="범위 밖", tripwire_triggered=bad)   # bad가 true가 되면 GuardrailFunctionOutput을 통해 범위 밖으로 출력

라우터는 질문 종류에 따라 두 에이전트 중 하나에 넘긴다.

In [16]:
reader = Agent(name="reader", model=MODEL, instructions="문서 내용을 묻는 질문만 맡는다.",
               tools=[grep_docs_tool])
router = Agent(name="router", model=MODEL,
               instructions="세는 질문은 counter 에게, 문서 내용 질문은 reader 에게 넘긴다.",
               handoffs=[counter, reader], input_guardrails=[scope_guard])

세는 질문을 라우터에 준다. 어느 에이전트가 답했는지 `last_agent`로 본다.

In [ ]:
out = await Runner.run(router, TASK)
print(out.last_agent.name, "|", out.final_output)

counter | 16


범위 밖 질문은 가드레일이 예외로 막는다.

In [18]:
try:
    await Runner.run(router, "오늘 서울 날씨는 어떤가")
except InputGuardrailTripwireTriggered:
    print("가드레일 | 차단됨")

가드레일 | 차단됨


### 선택 기준

| 조건 | 선택 |
|---|---|
| 반복 횟수만 제한하면 된다 | SDK |
| 컨텍스트 절단이나 압축이 필요하다 | 직접 구현 |
| 도구 단위 승인이 필요하다 | 직접 구현 |
| 평가 코드에 맞는 스팬이 필요하다 | 직접 구현 |
| 여러 팀이 표준 구조를 공유해야 한다 | SDK |
| 실패 형식을 자체 러너에 맞춰야 한다 | 변환 계층 추가 |

프레임워크가 감춘 부분이 요구사항과 충돌할 때만 해당 부분을 직접 구현한다.
+) 다른 종류의 LLM인 경우 호환이 되지 않음

### 비용 확인

`keep`으로 모아 둔 usage에 단가를 곱해 합산한다. SDK 쪽 사용량은 `keep`을 지나지 않으므로 여기에 없다. 그것도 감춰진 것 하나다.

In [ ]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")